# 02 — Preprocessing Pipeline
## Cleaning, Encoding, Normalization, SMOTE

This notebook runs the full preprocessing pipeline across all three datasets:
1. Clean sentinel values and impute missing data
2. Label-encode categorical features
3. StandardScaler normalization
4. Stratified train/test split (80/20)
5. SMOTE oversampling on training data only

Produces: V7 (SMOTE before/after), V8 (FL client distribution)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.preprocess import (
    load_and_clean_ulb, load_and_clean_baf, load_and_clean_synthetic,
    preprocess_dataset
)
from src.data.partition import partition_natural, save_client_data

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
FEDERATED_DIR = Path('../data/federated')

for d in [FIGURES_DIR, TABLES_DIR, PROCESSED_DIR, FEDERATED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load and Clean Datasets

In [ ]:
# ULB
print('=== ULB Dataset ===')
df_ulb, target_ulb = load_and_clean_ulb(RAW_DIR / 'ulb_creditcard.csv')
print(f'Shape: {df_ulb.shape}, Target: {target_ulb}')
print(f'Fraud rate: {df_ulb[target_ulb].mean()*100:.3f}%')

# BAF
print('\n=== BAF Dataset ===')
df_baf, target_baf = load_and_clean_baf(RAW_DIR / 'baf_base.csv')
print(f'Shape: {df_baf.shape}, Target: {target_baf}')
print(f'Fraud rate: {df_baf[target_baf].mean()*100:.3f}%')

# Synthetic (sampled)
print('\n=== Synthetic Dataset ===')
df_synth, target_synth = load_and_clean_synthetic(
    RAW_DIR / 'synthetic_fraud.csv', sample_size=500_000
)
print(f'Shape: {df_synth.shape}, Target: {target_synth}')
print(f'Fraud rate: {df_synth[target_synth].mean()*100:.3f}%')

## 2. Preprocess Each Dataset

In [ ]:
print('=== Preprocessing ULB ===')
X_train_ulb, X_test_ulb, y_train_ulb, y_test_ulb, scaler_ulb, enc_ulb = \
    preprocess_dataset(df_ulb, target_ulb)

print('\n=== Preprocessing BAF ===')
X_train_baf, X_test_baf, y_train_baf, y_test_baf, scaler_baf, enc_baf = \
    preprocess_dataset(df_baf, target_baf)

print('\n=== Preprocessing Synthetic ===')
X_train_synth, X_test_synth, y_train_synth, y_test_synth, scaler_synth, enc_synth = \
    preprocess_dataset(df_synth, target_synth)

## 3. V7 — SMOTE Before/After Visualization

In [ ]:
# Compute original (pre-SMOTE) train distributions
from sklearn.model_selection import train_test_split

original_counts = {
    'ULB': dict(pd.Series(df_ulb['Class']).value_counts()),
    'BAF': dict(pd.Series(df_baf['fraud_bool']).value_counts()),
    'Synthetic': dict(pd.Series(df_synth['is_fraud']).value_counts()),
}

smote_counts = {
    'ULB': dict(pd.Series(y_train_ulb).value_counts()),
    'BAF': dict(pd.Series(y_train_baf).value_counts()),
    'Synthetic': dict(pd.Series(y_train_synth).value_counts()),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, name in enumerate(['ULB', 'BAF', 'Synthetic']):
    # Before SMOTE
    ax = axes[0, i]
    orig = original_counts[name]
    bars = ax.bar(['Legitimate', 'Fraud'], [orig.get(0, 0), orig.get(1, 0)],
                  color=['#3498db', '#c0392b'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} — Before SMOTE')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

    # After SMOTE
    ax = axes[1, i]
    smote = smote_counts[name]
    bars = ax.bar(['Legitimate', 'Fraud'], [smote.get(0, 0), smote.get(1, 0)],
                  color=['#3498db', '#c0392b'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} — After SMOTE')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Class Distribution Before and After SMOTE', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'smote_before_after.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: smote_before_after.png')

## 4. Save Processed Data

In [ ]:
import joblib

datasets = {
    'ulb': (X_train_ulb, X_test_ulb, y_train_ulb, y_test_ulb, scaler_ulb, enc_ulb),
    'baf': (X_train_baf, X_test_baf, y_train_baf, y_test_baf, scaler_baf, enc_baf),
    'synthetic': (X_train_synth, X_test_synth, y_train_synth, y_test_synth, scaler_synth, enc_synth),
}

for name, (X_tr, X_te, y_tr, y_te, scaler, encoders) in datasets.items():
    prefix = PROCESSED_DIR / name
    X_tr.to_csv(f'{prefix}_X_train.csv', index=False)
    X_te.to_csv(f'{prefix}_X_test.csv', index=False)
    pd.Series(y_tr).to_csv(f'{prefix}_y_train.csv', index=False)
    pd.Series(y_te).to_csv(f'{prefix}_y_test.csv', index=False)
    joblib.dump(scaler, f'{prefix}_scaler.joblib')
    joblib.dump(encoders, f'{prefix}_encoders.joblib')
    print(f'Saved {name}: X_train={X_tr.shape}, X_test={X_te.shape}')

## 5. FL Data Partitioning (Natural Strategy)

Each dataset represents a different financial institution (client) in the FL setup.
Since datasets have different feature spaces, we'll handle feature alignment during FL.

In [ ]:
# Create natural FL partition (each dataset = one client)
clients = partition_natural(datasets)

# V8 — FL client data distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, client in enumerate(clients):
    ax = axes[i]
    counts = pd.Series(client['y_train']).value_counts().sort_index()
    bars = ax.bar(['Legitimate', 'Fraud'], counts.values,
                  color=['#3498db', '#c0392b'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'Client {i+1}: {client["name"].upper()}')
    ax.set_ylabel('Training Samples')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:,}', ha='center', va='bottom', fontsize=8)

fig.suptitle('FL Client Data Distribution (Natural Partition)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fl_client_distribution.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: fl_client_distribution.png')

In [ ]:
# Save FL client data
save_client_data(clients, output_dir=str(FEDERATED_DIR))

print('\n=== Preprocessing Complete ===')
print(f'Processed data saved to: {PROCESSED_DIR}')
print(f'FL client data saved to: {FEDERATED_DIR}')